#Evaluation, Data, Ethics & Recent Trends

In [4]:
!pip install -q "pyautogen<0.4" tavily-python python-dotenv numpy==1.26.4

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "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")  # <-- paste key here if not using .env

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file or paste it above.")

# DeepEval uses the OpenAI-compatible endpoint — we point it to Groq
os.environ["OPENAI_API_KEY"]  = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["GROQ_API_KEY"]    = GROQ_API_KEY

print(f"GROQ_API_KEY: set ✓")


GROQ_API_KEY: set ✓


# PART 2

In [8]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers langchain-groq

In [10]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Build a small knowledge base ─────────────────────────────────────────────
KNOWLEDGE_BASE = """
Large Language Models (LLMs) are neural networks trained on massive text corpora.
They learn to predict the next token in a sequence, which enables them to generate
coherent, contextually appropriate text.

The Transformer architecture, introduced in the 2017 paper "Attention Is All You Need"
by Vaswani et al., is the foundation of modern LLMs. Transformers use self-attention
mechanisms to weigh the importance of different tokens when generating each output token.

GPT-4, developed by OpenAI, is a multimodal LLM released in March 2023. It can process
both text and images as input. GPT-4 demonstrated significant improvements over GPT-3.5
on professional benchmarks, scoring in the top 10% on the bar exam.

Retrieval-Augmented Generation (RAG) is a technique that combines LLMs with a retrieval
system. Instead of relying solely on parametric knowledge (memorized during training),
RAG retrieves relevant documents at inference time and provides them as context to the LLM.
This reduces hallucination and allows the model to answer questions about recent events.

Hallucination in LLMs refers to the generation of factually incorrect or fabricated
information. It occurs because LLMs generate statistically likely text rather than
verifying facts. RAG and Retrieval Augmentation are common mitigation techniques.

Fine-tuning is the process of continuing to train a pre-trained LLM on a smaller,
task-specific dataset. It adjusts the model's weights to improve performance on
a particular task or domain. LoRA (Low-Rank Adaptation) is a parameter-efficient
fine-tuning method that only trains a small number of additional parameters.

Prompt engineering is the practice of crafting inputs (prompts) to guide LLM behavior.
Techniques include zero-shot prompting, few-shot prompting, chain-of-thought prompting,
and system prompts. Good prompt engineering can significantly improve output quality
without any model retraining.
"""

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents([KNOWLEDGE_BASE])
print(f"Knowledge base split into {len(chunks)} chunks.")

KeyError: '__reduce_cython__'

In [ ]:
# Build FAISS vector store with HuggingFace embeddings
print("Building FAISS vector store...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Build RAG chain
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, groq_api_key=GROQ_API_KEY)

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

In [ ]:
# ── Generate responses for evaluation ────────────────────────────────────────
test_questions = [
    "What is RAG and why does it reduce hallucination?",
    "Who invented the Transformer architecture?",
    "What is LoRA and how does it relate to fine-tuning?",
    "What score did GPT-4 achieve on the bar exam?",
    "What is the capital of France?"  # not in knowledge base — should trigger "I don't know"
]

rag_responses = {}
rag_contexts  = {}

print("Generating RAG responses...\n")
for q in test_questions:
    # Get retrieved docs separately (for evaluation context)
    docs = retriever.invoke(q)
    context_texts = [doc.page_content for doc in docs]
    response = rag_chain.invoke(q)

    rag_responses[q] = response
    rag_contexts[q]  = context_texts

    print(f"Q: {q}")
    print(f"A: {response[:200]}")
    print()

##PART 2.1

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.models import DeepEvalBaseLLM
from langchain_groq import ChatGroq

# ── Custom Groq model wrapper for DeepEval ────────────────────────────────────
# DeepEval needs an LLM to act as judge. We wrap Groq in its interface.
class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.client = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0,
            groq_api_key=GROQ_API_KEY
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        return self.client.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.client.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "groq/llama-3.3-70b-versatile"


judge_llm = GroqJudge()
print("DeepEval judge LLM ready.")

In [ ]:
# ── Build LLMTestCases ────────────────────────────────────────────────────────
test_cases = []
for q in test_questions:
    test_cases.append(
        LLMTestCase(
            input=q,
            actual_output=rag_responses[q],
            retrieval_context=rag_contexts[q]
        )
    )

print(f"Created {len(test_cases)} test cases.")

In [ ]:
# ── Run Metrics ───────────────────────────────────────────────────────────────
answer_relevancy = AnswerRelevancyMetric(threshold=0.7, model=judge_llm, include_reason=True)
faithfulness     = FaithfulnessMetric(threshold=0.7, model=judge_llm, include_reason=True)

print("Running DeepEval metrics (this calls the judge LLM for each test case)...")
print("=" * 60)

results = []
for i, tc in enumerate(test_cases):
    answer_relevancy.measure(tc)
    faithfulness.measure(tc)

    results.append({
        "question": tc.input,
        "answer": tc.actual_output[:120] + "...",
        "relevancy_score":    round(answer_relevancy.score, 3),
        "relevancy_reason":   answer_relevancy.reason,
        "faithfulness_score": round(faithfulness.score, 3),
        "faithfulness_reason": faithfulness.reason,
        "relevancy_passed":   answer_relevancy.is_successful(),
        "faithfulness_passed": faithfulness.is_successful()
    })

print("Evaluation complete.")